# Train "Hey Zen" wake word on MI300X

Runs the openWakeWord training pipeline (see `training/wakeword/README.md`) entirely inside this notebook's Jupyter session on the MI300X host — no SSH access needed from outside.

This exists because the same pipeline was tried on the dev laptop's gfx902 iGPU first and was projected to take **days** (clip generation alone was pacing at ~2–2.5 min per 25-clip batch, with 2000 batches needed). MI300X is gfx942, officially ROCm-supported — no `HSA_OVERRIDE_GFX_VERSION` override needed, and it should chew through this in minutes, not days.

**Before running:** confirm this Jupyter kernel has internet access (to clone the repo, pull the Piper TTS model, and pull ~7.5GB of ACAV100M negative-sample features from the internet) and ~20GB of free disk.

Run the cells top to bottom. Each setup cell is idempotent — safe to re-run if the kernel restarts partway through.

## 1. Confirm the GPU is visible

In [ ]:
!rocm-smi

## 2. Clone the repo

Public repo — no token needed. Only `training/wakeword/` is used from here on.

In [ ]:
import os
REPO_DIR = os.path.expanduser("~/candybot")
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/skariapaul/candybot.git {REPO_DIR}
else:
    print(f"Already cloned at {REPO_DIR} -- pulling latest")
    !git -C {REPO_DIR} pull
TRAINER_DIR = os.path.join(REPO_DIR, "training/wakeword/trainer")
print("TRAINER_DIR =", TRAINER_DIR)

## 3. Get Python 3.11

`piper-phonemize` (a training-only dependency) has no Linux wheel for Python 3.12+, only up to 3.11. Uses a portable build if 3.11 isn't already on this system — no sudo needed.

In [ ]:
import subprocess, shutil

py311 = shutil.which("python3.11")
if py311:
    print("Using system python3.11:", py311)
else:
    standalone_dir = os.path.expanduser("~/.local/python-standalone")
    py311 = os.path.join(standalone_dir, "python/bin/python3.11")
    if not os.path.exists(py311):
        os.makedirs(standalone_dir, exist_ok=True)
        tarball = os.path.join(standalone_dir, "cpython311.tar.gz")
        !curl -sL -o {tarball} "https://github.com/astral-sh/python-build-standalone/releases/latest/download/cpython-3.11.15+20260718-x86_64-unknown-linux-gnu-install_only.tar.gz"
        !tar -xzf {tarball} -C {standalone_dir}
    print("Using standalone python3.11:", py311)

assert os.path.exists(py311), "python3.11 not found after setup"
%env PY311={py311}

## 4. Create the training venv and install dependencies

In [ ]:
%%bash -s "$TRAINER_DIR"
set -euo pipefail
cd "$1"
if [ ! -d .venv-wakeword ]; then
  "$PY311" -m venv .venv-wakeword
fi
source .venv-wakeword/bin/activate
# CC/CXX override: the standalone Python build defaults to clang for
# compiling extensions (webrtcvad has one); most hosts only have gcc.
CC=gcc CXX=g++ pip install -q -r requirements.txt
echo "base requirements installed"

In [ ]:
%%bash -s "$TRAINER_DIR"
set -euo pipefail
cd "$1"
source .venv-wakeword/bin/activate
# requirements.txt pulls the default CUDA-oriented PyPI build of torch,
# which won't see this AMD GPU -- swap in the ROCm wheel. MI300X = gfx942,
# officially supported, no HSA_OVERRIDE_GFX_VERSION workaround needed
# (unlike the dev laptop's gfx902 iGPU).
pip uninstall -y -q torch torchaudio
pip install -q torch==2.7.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/rocm6.3
python -c "import torch; print('CUDA/ROCm available:', torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'n/a')"

Expect `CUDA/ROCm available: True` and the MI300X's device name printed above — if not, stop here and debug before continuing (the rest of this notebook assumes GPU is working).

## 5. Run the full pipeline

All 11 steps: download datasets → verify → resolve config → generate TTS clips → augment → train → export. This cell blocks until done or until a step fails.

If it fails partway or the kernel restarts, re-run with `--from <step>` (the error message names the exact step) instead of re-running this cell from scratch — see `trainer/README.md`'s CLI reference.

In [ ]:
%%bash -s "$TRAINER_DIR"
set -euo pipefail
cd "$1"
source .venv-wakeword/bin/activate
python train_wakeword.py --config configs/hey_zen.yaml

## 6. Verify the exported model

In [ ]:
%%bash -s "$TRAINER_DIR"
cd "$1"
ls -la export/hey_zen.onnx export/hey_zen.onnx.data

## 7. Get the model back to the dev laptop

Both export files together are ~200KB — small enough to just commit and push straight from here, using whatever git credentials are already configured in this Jupyter environment (a GitHub token / SSH key you set up on this MI300X host yourself — not something this notebook manages).

Skip this cell and download the two files through Jupyter's file browser instead if you'd rather not push from here.

In [ ]:
%%bash -s "$REPO_DIR"
set -euo pipefail
cd "$1"
git add training/wakeword/trainer/export/hey_zen.onnx training/wakeword/trainer/export/hey_zen.onnx.data
git commit -m "Add trained hey_zen wake-word model (MI300X)"
git push origin main

## Next step (back on the dev laptop)

```bash
git pull
cp training/wakeword/trainer/export/hey_zen.onnx* candybot/models/wakeword/
```
Then set `configs/candybot.yaml`'s `voice.wake_word.model` to `"hey_zen"`. See `training/wakeword/README.md`'s "After training" section.